**Group Kappa Chungus Deluxe**



USE DINOv3 


In [ ]:
# If you are running in a fresh environment, uncomment to install dependencies:
# !pip -q install torch torchvision transformers timm scikit-learn matplotlib pandas tqdm pillow

import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import pandas as pd




print("torch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
from huggingface_hub import login; login(token="hf_****")

def set_seed(seed: int = 42):
    """Make results as reproducible as possible across runs."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(42)


In [ ]:
#build data pipeline
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoImageProcessor


MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"
TRAIN_DIR = "./train"
TEST_DIR = "./test"
NUM_CLASSES = 100
IMG_SIZE = 224
batch_size = 32


processor = AutoImageProcessor.from_pretrained(MODEL_ID)
MEAN = getattr(processor, "image_mean", [0.485, 0.456, 0.406])
STD = getattr(processor, "image_std", [0.229, 0.224, 0.225])
print("normalize mean/std:", MEAN, STD)



train_tf = transforms.Compose([transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.25),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

class TheDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]

class TestDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), os.path.basename(self.paths[i])

train_paths = []
train_labels = []


for cls in sorted(os.listdir(TRAIN_DIR), key=lambda x: int(x)):
    for p in glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg")):
        train_paths.append(p)
        train_labels.append(int(cls))
train_paths = np.array(train_paths)
train_labels = np.array(train_labels)

test_paths = sorted(glob.glob(os.path.join(TEST_DIR, "*.jpg")), key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))



print("train images:", len(train_paths), "| classes:", len(np.unique(train_labels)))
#print("")
print("test images:", len(test_paths))
print("min/max per class:", np.bincount(train_labels).min(), np.bincount(train_labels).max())


load DINOv3 ViT-L/16, freeze it, then get  **CLS token**

In [ ]:
#b ackbone, head, and precomputed embeddings

from transformers import AutoModel


backbone = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
for p in backbone.parameters():
    p.requires_grad_(False)
EMB_DIM = backbone.config.hidden_size
print("backbone hidden size:", EMB_DIM)

@torch.no_grad()
def extract(images):
    out = backbone(pixel_values=images.to(device))
    return out.last_hidden_state[:, 0]



class ClassiferHead(nn.Module):
    def __init__(self, in_dim=EMB_DIM, hidden=512, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x):
        return self.net(x)



@torch.no_grad()
def embed_dataset(ds, shuffle=False):
    loader = DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)
    embs = []
    for imgs, _ in tqdm(loader, desc="embedding", leave=False):
        embs.append(extract(imgs).cpu())
    return torch.cat(embs)

#validation
train_emb_clean = embed_dataset(TheDataset(train_paths, train_labels, eval_tf))
test_emb = embed_dataset(TestDataset(test_paths, eval_tf))
labels_t = torch.tensor(train_labels, dtype=torch.long)



#CHANGE THIS?!
NUM_VIEWS = 8


train_emb_aug = torch.stack(
    [embed_dataset(TheDataset(train_paths, train_labels, train_tf))
        for _ in tqdm(range(NUM_VIEWS), desc="aug views")],
    dim=1,
)

print("train_emb_clean:", tuple(train_emb_clean.shape), "| test_emb:", tuple(test_emb.shape))
print("train_emb_aug:", tuple(train_emb_aug.shape), f"({train_emb_aug.element_size() * train_emb_aug.nelement() / 1e6:.1f} MB)")



TRAIN FUNCTION HERE. take random stuff from image every epoch (N K D) 

In [ ]:
#train here

from timm.loss import SoftTargetCrossEntropy


train_criterion = SoftTargetCrossEntropy()
val_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


#can be changed
MIXUP_ALPHA = 0.2
LABEL_SMOOTHING = 0.1

class CachedViewDataset(Dataset):
    def __init__(self, bank, labels):
        self.bank = bank
        self.labels = labels
        self.K = bank.size(1)
    def __len__(self):
        return self.bank.size(0)
    def __getitem__(self, i):
        k = torch.randint(self.K, (1,)).item()
        return self.bank[i, k], self.labels[i]

def feature_mixup(emb, targets, num_classes=NUM_CLASSES, alpha=MIXUP_ALPHA, smoothing=LABEL_SMOOTHING):
    off = smoothing / num_classes
    y = torch.full((targets.size(0), num_classes), off, device=emb.device)
    y.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing + off)
    if alpha <= 0:
        return emb, y
    
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(emb.size(0), device=emb.device)

    
    emb = lam * emb + (1.0 - lam) * emb[perm]
    y = lam * y + (1.0 - lam) * y[perm]
    return emb, y


#avg loss here
def train_one_epoch(head, optimizer, loader):
    head.train()
    total_loss = 0.0
    total = 0
    for emb, targets in loader:
        emb, targets = emb.to(device), targets.to(device)
        emb, soft_targets = feature_mixup(emb, targets)
        optimizer.zero_grad()
        loss = train_criterion(head(emb), soft_targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * emb.size(0)
        total += emb.size(0)

    
    return total_loss / total


In [ ]:
#validation fucntion 
@torch.no_grad()
def evaluate(head, emb, targets):
    head.eval()
    emb = emb.to(device)
    targets = targets.to(device)
    logits = head(emb)
    loss = val_criterion(logits, targets).item()
    preds = logits.argmax(dim=1)
    acc = (preds == targets).float().mean().item()
    return loss, acc, preds.cpu()


train with k fold here



In [ ]:

import copy
from sklearn.model_selection import StratifiedKFold

K = 5
EPOCHS = 60 
PATIENCE = 10
LR = 1e-3
WEIGHT_DECAY = 0.05

ckpt_dir = "./checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)
fold_histories = []
fold_best_states = []
fold_best_acc = []
oof_preds = np.full(len(train_labels), -1, dtype=int)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_paths, train_labels), start=1):

    
    tr_ds = CachedViewDataset(train_emb_aug[tr_idx], labels_t[tr_idx])
    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)
    va_emb = train_emb_clean[va_idx]
    va_lab = labels_t[va_idx]

    head = ClassiferHead().to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    hist = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_loss, best_acc, best_state, best_preds, wait = float("inf"), 0.0, None, None, 0

    for epoch in range(1, EPOCHS + 1):
        tr_loss = train_one_epoch(head, optimizer, tr_loader)
        va_loss, va_acc, va_pred = evaluate(head, va_emb, va_lab)
        scheduler.step()

        hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(va_loss)
        hist["val_acc"].append(va_acc)

        #stop it here 
        if va_loss < best_loss:
            best_loss, wait = va_loss, 0
        else:
            wait += 1
        if va_acc > best_acc:
            best_acc = va_acc
            best_state = copy.deepcopy(head.state_dict())
            best_preds = va_pred.numpy()
        if wait >= PATIENCE:
            print(f"early stop at epoch {epoch}")
            break

    print(f"fold {fold}/{K} | best val_acc={best_acc:.4f} | epochs={len(hist['train_loss'])}")
    torch.save(best_state, os.path.join(ckpt_dir, f"head_fold{fold}.pt"))
    fold_histories.append(hist)
    fold_best_states.append(best_state)
    fold_best_acc.append(best_acc)
    oof_preds[va_idx] = best_preds

print(f"\nmean CV val_acc: {np.mean(fold_best_acc):.4f} +/- {np.std(fold_best_acc):.4f}")


In [ ]:
#
# kfold jhere# 
import copy
from sklearn.model_selection import StratifiedKFold

K = 5

EPOCHS = 60
PATIENCE = 10
LR = 1e-3

#here? change this
WEIGHT_DECAY = 0.05 

ckpt_dir = "./checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)
fold_history = []
fold_best_states = []
fold_best_acc = []
oof_preds = np.full(len(train_labels), -1, dtype=int)


for fold, (tr_idx, va_idx) in enumerate(skf.split(train_paths, train_labels), start=1):


    tr_ds = CachedViewDataset(train_emb_aug[tr_idx], labels_t[tr_idx])
    tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)


    va_emb = train_emb_clean[va_idx]
    va_lab = labels_t[va_idx]
    head = ClassiferHead().to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    hist = {"train_loss": [], "val_loss": [], "val_acc": []}

    best_loss = float("inf")
    best_acc = 0.0
    best_state = None
    best_preds = None
    wait = 0

    for epoch in range(1, EPOCHS + 1):
        tr_loss = train_one_epoch(head, optimizer, tr_loader)
        va_loss, va_acc, va_pred = evaluate(head, va_emb, va_lab)
        scheduler.step()

        hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(va_loss)
        hist["val_acc"].append(va_acc)
        if va_loss < best_loss:
            best_loss, wait = va_loss, 0
        else:
            wait += 1
        if va_acc > best_acc:
            best_acc = va_acc
            best_state = copy.deepcopy(head.state_dict())
            best_preds = va_pred.numpy()
        if wait >= PATIENCE:
            print(f"stopped early at epoch {epoch}")
            break

    print(f"fold {fold}/{K} | best val_acc={best_acc:.4f} | epochs={len(hist['train_loss'])}")
    torch.save(best_state, os.path.join(ckpt_dir, f"head_fold{fold}.pt"))
    fold_history.append(hist)
    fold_best_states.append(best_state)
    fold_best_acc.append(best_acc)
    oof_preds[va_idx] = best_preds

print(f"\nMean CV val_acc: {np.mean(fold_best_acc):.4f} +/- {np.std(fold_best_acc):.4f}")


PLOT HERE

In [ ]:


fig, axes = plt.subplots(1, 2, figsize=(16, 5))


for f, h in enumerate(fold_history, start=1):
    ep = range(1, len(h["train_loss"]) + 1)
    axes[0].plot(ep, h["train_loss"], linestyle="--", alpha=0.5)
    axes[0].plot(ep, h["val_loss"], marker="o", label=f"Fold {f} val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss per fold (dashed=train, solid=val)")
axes[0].legend(fontsize=8)
axes[0].grid(True)



for f, h in enumerate(fold_history, start=1):
    ep = range(1, len(h["val_acc"]) + 1)
    axes[1].plot(ep, h["val_acc"], marker="o", label=f"Fold {f}")
axes[1].axhline(np.mean(fold_best_acc), color="k", linestyle=":", label=f"Mean best={np.mean(fold_best_acc):.3f}")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation Accuracy")
axes[1].set_title("Validation Accuracy per fold")
axes[1].legend(fontsize=8)
axes[1].grid(True)

#plt
plt.tight_layout()
plt.show()


make csv file here


In [ ]:
#submsison file
@torch.no_grad()
def predict_proba(state, emb):
    head = ClassiferHead().to(device)
    head.load_state_dict(state)
    head.eval()
    return torch.softmax(head(emb.to(device)), dim=1).cpu()
probs = torch.zeros(len(test_paths), NUM_CLASSES)
for state in fold_best_states:
    probs += predict_proba(state, test_emb)
probs /= len(fold_best_states)

pred_labels = probs.argmax(dim=1).numpy()
test_ids = [os.path.basename(p) for p in test_paths]

submission = pd.DataFrame({"ID": test_ids, "Label": pred_labels})
submission.to_csv("submission.csv", index=False)
print("wrote submission.csv:", submission.shape)
submission.head()
